# 01 - Exploracao Inicial

Analise inicial da base bruta de vendas de sorvetes de 2025.

**Escopo desta etapa**

- Ler o CSV bruto em `data/raw/vendas_sorvetes.csv`.
- Consultar o contexto do case em `docs/case_sorveteria.pptx`.
- Mapear estrutura, tipos, nulos, duplicidades, estatisticas descritivas e problemas provaveis de qualidade.
- Nao alterar arquivos em `data/raw`.
- Nao fazer limpeza pesada, dashboard ou KPIs finais.

In [1]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "vendas_sorvetes.csv"
PPTX_PATH = PROJECT_ROOT / "docs" / "case_sorveteria.pptx"

RAW_DATA_PATH, PPTX_PATH

(WindowsPath('C:/Users/User/OneDrive/Documentos/David/case-sorveteria-analytics/data/raw/vendas_sorvetes.csv'),
 WindowsPath('C:/Users/User/OneDrive/Documentos/David/case-sorveteria-analytics/docs/case_sorveteria.pptx'))

## Contexto Do Case

O arquivo de apresentacao foi usado apenas para recuperar o contexto e o dicionario informado no briefing.

In [2]:
def extract_pptx_text(path: Path) -> pd.DataFrame:
    ns = {"a": "http://schemas.openxmlformats.org/drawingml/2006/main"}
    records = []

    with zipfile.ZipFile(path) as zf:
        slide_names = sorted(
            name for name in zf.namelist()
            if name.startswith("ppt/slides/slide") and name.endswith(".xml")
        )
        for index, slide_name in enumerate(slide_names, start=1):
            tree = ET.fromstring(zf.read(slide_name))
            text = " ".join(node.text.strip() for node in tree.findall(".//a:t", ns) if node.text)
            records.append({"slide": index, "texto": text})

    return pd.DataFrame(records)

pptx_text = extract_pptx_text(PPTX_PATH)
pptx_text

,slide,texto
0,1,Case Shell Box Sorveteria *Dados fictícios par...
1,2,Entregável Nosso time de analytics foi acionad...


## Leitura Da Base Bruta

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

linhas, colunas = df.shape
print(f"Linhas: {linhas:,}".replace(",", "."))
print(f"Colunas: {colunas}")

Linhas: 50.000
Colunas: 12


## Primeiras Linhas

In [4]:
df.head()

,ID_Transacao,Data,Hora,Tipo_Sorvete,Sabor,Quantidade,Valor_Total,Cidade,Estado,Canal_Venda,Promocao,ID_Cliente
0,1,2025-04-30,11:00,Milkshake,Açaí,3,16.22,Campinas,AP,App,True,CLI9795
1,2,2025-07-12,15:45,Milkshake,Menta,4,18.57,Aragão,TO,App,True,CLI1799
2,3,2025-03-08,15:15,Pote,Baunilha,4,21.76,Câmara,PA,Parceiro,False,CLI9914
3,4,2025-08-13,21:45,Casquinha,Cookies,3,30.87,Carvalho de Pereira,PR,Loja Física,True,CLI2071
4,5,2025-04-07,11:45,Picolé,Cookies,4,26.59,Moraes,MA,Loja Física,True,CLI9549


## Colunas E Tipos De Dados

In [5]:
tipos = (
    pd.DataFrame({
        "coluna": df.columns,
        "tipo_identificado": [str(dtype) for dtype in df.dtypes],
        "valores_nao_nulos": df.notna().sum().values,
        "valores_unicos": df.nunique(dropna=True).values,
    })
)
tipos

,coluna,tipo_identificado,valores_nao_nulos,valores_unicos
0,ID_Transacao,int64,50000,50000
1,Data,str,50000,213
2,Hora,str,50000,60
3,Tipo_Sorvete,str,50000,5
4,Sabor,str,49500,8
5,Quantidade,int64,50000,13
6,Valor_Total,float64,49500,6786
7,Cidade,str,49500,6554
8,Estado,str,50000,27
9,Canal_Venda,str,50000,3


## Valores Nulos

In [6]:
nulos = (
    pd.DataFrame({
        "coluna": df.columns,
        "nulos": df.isna().sum().values,
        "percentual_nulos": (df.isna().mean().values * 100).round(2),
    })
    .sort_values(["nulos", "coluna"], ascending=[False, True])
)
nulos

,coluna,nulos,percentual_nulos
7,Cidade,500,1.00
4,Sabor,500,1.00
6,Valor_Total,500,1.00
9,Canal_Venda,0,0.00
1,Data,0,0.00
8,Estado,0,0.00
2,Hora,0,0.00
11,ID_Cliente,0,0.00
0,ID_Transacao,0,0.00
10,Promocao,0,0.00


## Duplicidades

In [7]:
duplicidades = pd.DataFrame([
    {"checagem": "linhas totalmente duplicadas", "quantidade": int(df.duplicated().sum())},
    {"checagem": "ID_Transacao duplicado", "quantidade": int(df["ID_Transacao"].duplicated().sum())},
    {"checagem": "ID_Cliente distintos", "quantidade": int(df["ID_Cliente"].nunique())},
])
duplicidades

,checagem,quantidade
0,linhas totalmente duplicadas,0
1,ID_Transacao duplicado,0
2,ID_Cliente distintos,8974


## Estatisticas Descritivas

In [8]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID_Transacao,"50,000.00",NaN,NaN,NaN,"25,000.50","14,433.90",1.00,"12,500.75","25,000.50","37,500.25","50,000.00"
Data,50000,213,2025-05-04,315,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hora,50000,60,19:30,919,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tipo_Sorvete,50000,5,Milkshake,11959,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sabor,49500,8,Açaí,6256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantidade,"50,000.00",NaN,NaN,NaN,2.98,1.59,-6.00,2.00,3.00,4.00,6.00
Valor_Total,"49,500.00",NaN,NaN,NaN,27.28,17.41,-89.82,13.84,25.23,38.44,89.88
Cidade,49500,6554,Campinas,2461,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Estado,50000,27,AL,1969,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Canal_Venda,50000,3,Parceiro,16771,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Checagens De Datas, Horarios E Valores

In [9]:
datas = pd.to_datetime(df["Data"], errors="coerce")
horas = pd.to_datetime(df["Hora"], format="%H:%M", errors="coerce")

checagens = pd.DataFrame([
    {"item": "datas invalidas", "resultado": int(datas.isna().sum())},
    {"item": "data minima", "resultado": datas.min().date()},
    {"item": "data maxima", "resultado": datas.max().date()},
    {"item": "anos encontrados", "resultado": sorted(datas.dt.year.dropna().astype(int).unique().tolist())},
    {"item": "horarios invalidos", "resultado": int(horas.isna().sum())},
    {"item": "quantidade <= 0", "resultado": int((df["Quantidade"] <= 0).sum())},
    {"item": "valor_total <= 0", "resultado": int((df["Valor_Total"] <= 0).sum())},
])
checagens

,item,resultado
0,datas invalidas,0
1,data minima,2025-02-20
2,data maxima,2025-09-20
3,anos encontrados,[2025]
4,horarios invalidos,0
5,quantidade <= 0,1016
6,valor_total <= 0,1009


## Distribuicao De Categorias

In [10]:
categoricas = ["Tipo_Sorvete", "Sabor", "Cidade", "Estado", "Canal_Venda", "Promocao"]

for coluna in categoricas:
    print(f"\n{coluna}")
    display(df[coluna].value_counts(dropna=False).head(10).to_frame("quantidade"))


Tipo_Sorvete


,quantidade
Tipo_Sorvete,
Milkshake,11959
Sundae,9576
Picolé,9527
Pote,9507
Casquinha,9431



Sabor


,quantidade
Sabor,
Açaí,6256
Cookies,6253
Caramelo,6232
Morango,6190
Menta,6184
Limão,6181
Baunilha,6164
Chocolate,6040
NaN,500



Cidade


...

,quantidade
Estado,
AL,1969
PE,1916
PR,1892
MG,1891
MS,1889
CE,1886
MA,1881
SP,1876
SC,1873



Canal_Venda


,quantidade
Canal_Venda,
Parceiro,16771
App,16636
Loja Física,16593



Promocao


,quantidade
Promocao,
True,25028
False,24972


## Campos Provaveis Por Papel Analitico

In [11]:
papeis = pd.DataFrame([
    {"papel": "data", "campos_provaveis": "Data, Hora"},
    {"papel": "produto", "campos_provaveis": "Tipo_Sorvete, Sabor"},
    {"papel": "canal", "campos_provaveis": "Canal_Venda"},
    {"papel": "receita", "campos_provaveis": "Valor_Total"},
    {"papel": "quantidade", "campos_provaveis": "Quantidade"},
    {"papel": "loja", "campos_provaveis": "nao identificado explicitamente"},
    {"papel": "regiao", "campos_provaveis": "Cidade, Estado"},
    {"papel": "vendedor", "campos_provaveis": "nao identificado explicitamente"},
    {"papel": "cliente", "campos_provaveis": "ID_Cliente"},
])
papeis

,papel,campos_provaveis
0,data,"Data, Hora"
1,produto,"Tipo_Sorvete, Sabor"
2,canal,Canal_Venda
3,receita,Valor_Total
4,quantidade,Quantidade
5,loja,nao identificado explicitamente
6,regiao,"Cidade, Estado"
7,vendedor,nao identificado explicitamente
8,cliente,ID_Cliente


## Possiveis Problemas De Qualidade

- `Sabor`, `Valor_Total` e `Cidade` possuem valores nulos, cada um com 500 registros.
- `Quantidade` possui valores menores ou iguais a zero; isso pode indicar devolucao, cancelamento, erro de carga ou regra de negocio nao documentada.
- `Valor_Total` possui valores menores ou iguais a zero e tambem valores nulos; antes de receita ou ticket medio, e necessario definir como tratar esses casos.
- `Data` e `Hora` foram lidas como texto; devem ser convertidas em etapa futura de tratamento.
- `Cidade` tem alta cardinalidade e exemplos que podem exigir validacao de consistencia com `Estado`.
- Nao ha campo explicito de loja ou vendedor na base atual.

## Conclusao Inicial

A base esta pronta para uma etapa estruturada de qualidade e tratamento, mas ainda nao deve alimentar KPIs finais sem validar valores nulos, quantidades nao positivas, valores nao positivos e a consistencia geografica. A camada `data/raw` deve permanecer imutavel; qualquer correcao deve gerar novas bases em `data/interim` ou `data/processed`.